## Importando bibliotecas

In [92]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os

## Lendo os dados

In [93]:
food_app_path = os.path.join(os.getcwd(), '..', 'data', 'logs_exp_us.csv')
food_app = pd.read_csv(food_app_path, sep='\t')

## Explorando os dados

In [94]:
# Visão geral
print(food_app.info())
print('\n', food_app.head(10))

# Estatísticas descritivas
print('\nEstatísticas descritivas:')
print('\n', food_app.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244126 entries, 0 to 244125
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   EventName       244126 non-null  object
 1   DeviceIDHash    244126 non-null  int64 
 2   EventTimestamp  244126 non-null  int64 
 3   ExpId           244126 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 7.5+ MB
None

                  EventName         DeviceIDHash  EventTimestamp  ExpId
0         MainScreenAppear  4575588528974610257      1564029816    246
1         MainScreenAppear  7416695313311560658      1564053102    246
2  PaymentScreenSuccessful  3518123091307005509      1564054127    248
3         CartScreenAppear  3518123091307005509      1564054127    248
4  PaymentScreenSuccessful  6217807653094995999      1564055322    248
5         CartScreenAppear  6217807653094995999      1564055323    248
6       OffersScreenAppear  8351860793733343758      1564066242   

## Preparando dados para análise

### Verificação de valores duplicados e adequação dos nomes de colunas

In [95]:
# Adequação dos nomes das colunas
food_app = food_app.rename(columns={
    'EventName': 'event_name',      # nome do evento
    'DeviceIDHash': 'device_id',    # identificador de usuário exclusivo
    'EventTimestamp': 'timestamp',  # hora do evento
    'ExpId': 'exp_id'               # número do experimento: 246 e 247 são os grupos de controle, 248 é o grupo de teste
})

In [96]:
# Verificando valores duplicados
print('\n'+'Número de linhas duplicadas:')
print(food_app.duplicated().sum())


Número de linhas duplicadas:
413


In [97]:
# Visualizando as linhas duplicadas
dups_mask = food_app.duplicated(keep=False)  # exibe todas as ocorrências (original + cópia)
print(f'Linhas envolvidas em duplicações: {dups_mask.sum()}')
food_app[dups_mask].sort_values(list(food_app.columns)).head(10)

Linhas envolvidas em duplicações: 768


,event_name,device_id,timestamp,exp_id
104106,CartScreenAppear,34565258828294726,1564857221,248
104108,CartScreenAppear,34565258828294726,1564857221,248
17036,CartScreenAppear,197027893265565660,1564659614,246
17037,CartScreenAppear,197027893265565660,1564659614,246
23419,CartScreenAppear,197027893265565660,1564668928,246
23421,CartScreenAppear,197027893265565660,1564668928,246
34222,CartScreenAppear,197027893265565660,1564684544,246
34223,CartScreenAppear,197027893265565660,1564684544,246
112561,CartScreenAppear,197027893265565660,1564902904,246
112562,CartScreenAppear,197027893265565660,1564902904,246


In [98]:
# Impacto percentual dos duplicados no total do dataset
total = len(food_app)                   # total de linhas no dataset
n_dups = food_app.duplicated().sum()    # número de linhas duplicadas
pct = round(n_dups / total * 100, 2)    # percentual de linhas duplicadas
print(f'Total de linhas:     {total}')
print(f'Linhas duplicadas:   {n_dups}')
print(f'Impacto no dataset:  {pct}%')

Total de linhas:     244126
Linhas duplicadas:   413
Impacto no dataset:  0.17%


In [99]:
# Distribuição dos duplicados por tipo de evento
# Verificando se os duplicados estão concentrados em algum tipo de evento específico
dup_rows = food_app[food_app.duplicated(keep=False)]
print('Duplicatas por event_name:')
print(dup_rows.groupby('event_name').size().sort_values(ascending=False))

Duplicatas por event_name:
event_name
PaymentScreenSuccessful    348
MainScreenAppear           207
CartScreenAppear           126
Tutorial                    54
OffersScreenAppear          33
dtype: int64


In [100]:
# Duplicatas por grupo experimental (exp_id)
# Verificando se os duplicados estão concentrados em algum grupo específico
print('Duplicatas por exp_id:')
print(dup_rows.groupby('exp_id').size().sort_values(ascending=False))

Duplicatas por exp_id:
exp_id
248    310
247    233
246    225
dtype: int64


In [101]:
# Proporção de duplicatas relativa ao tamanho de cada grupo
# Necessário para entender se um grupo tem mais duplicatas proporcionalmente ao seu tamanho
total_per_group = food_app.groupby('exp_id').size().rename('total')
dups_per_group  = dup_rows.groupby('exp_id').size().rename('duplicatas')

resumo = pd.concat([total_per_group, dups_per_group], axis=1)
resumo['% duplicatas'] = (round(resumo['duplicatas'] / resumo['total'] * 100, 2).astype(str) + '%')
resumo.index.name = None  # alinhando o índice na mesma altura dos cabeçalhos

print('Proporção de duplicatas por grupo experimental:', '\n')
print(resumo.to_string())

Proporção de duplicatas por grupo experimental: 

     total  duplicatas % duplicatas
246  80304         225        0.28%
247  78075         233         0.3%
248  85747         310        0.36%


In [102]:
# Verificando se o mesmo device_id repete o evento no mesmo timestamp (duplicata verdadeira)
# Verificando se realmente é o mesmo usuário com o mesmo evento no mesmo segundo
print('Duplicatas por device_id + event_name + timestamp:')
print(food_app.duplicated(subset=['device_id', 'event_name', 'timestamp']).sum())

Duplicatas por device_id + event_name + timestamp:
413


In [103]:
# Removendo as linhas duplicadas
food_app_clean = food_app.drop_duplicates().copy()
print(f'Número de linhas após remoção de duplicatas: {len(food_app_clean)}')

Número de linhas após remoção de duplicatas: 243713


#### Conclusão de impacto 

- Como o percentual é pequeno (< 1-2%) e as duplicatas estão distribuídas uniformemente entre os grupos, o impacto é baixo e podem ser removidas com *drop_duplicates()*. 

- Caso estivessem concentradas em um grupo apenas, caberia investigar antes de remover.

### Adição de uma coluna de data e hora e uma coluna separada para datas

In [104]:
# Convertendo timestamp (int64 Unix) para datetime e extraindo a data
food_app_clean['datetime'] = pd.to_datetime(food_app_clean['timestamp'], unit='s')
food_app_clean['date']     = food_app_clean['datetime'].dt.date

print(food_app_clean[['timestamp', 'datetime', 'date']].head())
print()
print()
print(food_app_clean.info())

    timestamp            datetime        date
0  1564029816 2019-07-25 04:43:36  2019-07-25
1  1564053102 2019-07-25 11:11:42  2019-07-25
2  1564054127 2019-07-25 11:28:47  2019-07-25
3  1564054127 2019-07-25 11:28:47  2019-07-25
4  1564055322 2019-07-25 11:48:42  2019-07-25


<class 'pandas.core.frame.DataFrame'>
Index: 243713 entries, 0 to 244125
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   event_name  243713 non-null  object        
 1   device_id   243713 non-null  int64         
 2   timestamp   243713 non-null  int64         
 3   exp_id      243713 non-null  int64         
 4   datetime    243713 non-null  datetime64[ns]
 5   date        243713 non-null  object        
dtypes: datetime64[ns](1), int64(3), object(2)
memory usage: 13.0+ MB
None


## Estudo e verificação de dados

### Quantos eventos ficam nos registros?

In [105]:
# Cada linha do DataFrame representa um evento registrado no app
# len() retorna o total de linhas — equivalente ao total de eventos após a limpeza

total_events = len(food_app_clean)
print(f'Total de eventos nos registros: {total_events:,}')

Total de eventos nos registros: 243,713



>**Raciocínio:** Após a limpeza (remoção de duplicatas), é necessário confirmar o volume real de eventos com o qual trabalharemos nas análises seguintes. Usar `len()` sobre o DataFrame limpo é a forma mais direta — cada linha representa um evento registrado.

### Quantos usuários ficam nos registros?

In [106]:
# device_id identifica cada usuário de forma única
# nunique() conta apenas os valores distintos — ignora repetições do mesmo usuário

total_users = food_app_clean['device_id'].nunique()
print(f'Total de usuários nos registros: {total_users:,}')

Total de usuários nos registros: 7,551



>**Raciocínio:** O `device_id` é o identificador único de cada usuário. Um mesmo usuário pode gerar vários eventos, portanto ao utilizar `nunique()` teremos apenas os valores distintos, eliminando a repetição natural de um usuário que interagiu múltiplas vezes com o app.

### Qual é o número médio de eventos por usuário?

In [107]:
# Calculando quantos eventos cada usuário gerou individualmente
events_per_user = food_app_clean.groupby('device_id').size()

# Média simples: total de eventos dividido pelo total de usuários
mean_events = events_per_user.mean()

print(f'Média de eventos por usuário: {mean_events:.2f}')
print()

# describe() expõe a distribuição completa — essencial para detectar
# se a média está sendo puxada por usuários extremamente ativos (outliers)
print('Distribuição do número de eventos por usuário:')
print(events_per_user.describe().round(2))

Média de eventos por usuário: 32.28

Distribuição do número de eventos por usuário:
count    7551.00
mean       32.28
std        65.15
min         1.00
25%         9.00
50%        20.00
75%        37.00
max      2307.00
dtype: float64



>**Raciocínio:** A média simples (total de eventos ÷ total de usuários) fornece uma visão geral do engajamento médio. O uso de `groupby + describe()` possibilita uma percepção da distribuição completa — a média isolada pode ser enganosa se a distribuição for muito assimétrica (ex.: poucos usuários hiper-ativos distorcendo o valor). Ver mediana, mínimo, máximo e percentis permite um diagnóstico mais honesto do comportamento da base.

### Qual é o período de tempo que os dados cobrem?

In [108]:
# Data e hora mínima e máxima registradas no dataset limpo
data_min = food_app_clean['datetime'].min()
data_max = food_app_clean['datetime'].max()

# Diferença total em dias entre o primeiro e o último evento
total_period = (data_max - data_min).days

print(f'Data mínima : {data_min}')
print(f'Data máxima : {data_max}')
print(f'Período total: {total_period} dias')

Data mínima : 2019-07-25 04:43:36
Data máxima : 2019-08-07 21:15:17
Período total: 13 dias


> **Raciocínio:** O `datetime` já foi criado na etapa de preparação; basta aplicar `.min()` e `.max()` diretamente sobre essa coluna. A subtração entre os dois valores retorna um `Timedelta` do pandas — acessamos `.days` para ter o número inteiro de dias do intervalo.

### Histograma de eventos por data e hora

In [109]:
# Extração da hora do dia para o eixo de distribuição intradiária
food_app_clean['hour'] = food_app_clean['datetime'].dt.hour     # extraindo apenas a hora (0-23) de cada registro

# --- Histograma 1: eventos por dia ---
events_per_day = food_app_clean.groupby('date').size().reset_index(name='n_eventos')  # contando eventos agrupados por data
events_per_day['date'] = pd.to_datetime(events_per_day['date'])  # convertendo para datetime para ordenação correta no eixo x

fig_day = px.bar(
    events_per_day,
    x='date',           # eixo x: data
    y='n_eventos',      # eixo y: volume de eventos naquele dia
    title='Eventos por dia',
    labels={'date': 'Data', 'n_eventos': 'Número de eventos'}
)
fig_day.update_layout(bargap=0.1)  # espaçamento entre barras para melhor legibilidade
fig_day.show()

# --- Histograma 2: eventos por hora do dia ---
fig_hour = px.histogram(
    food_app_clean,
    x='hour',           # eixo x: hora do dia (0-23)
    nbins=24,           # um bin por hora — granularidade máxima sem perda de informação
    title='Distribuição de eventos por hora do dia',
    labels={'hour': 'Hora do dia', 'count': 'Número de eventos'}
)
fig_hour.update_layout(bargap=0.05, xaxis=dict(dtick=1))  # dtick=1 força exibição de todas as 24 horas no eixo x
fig_hour.show()

> **Raciocínio:** Dois histogramas complementares respondem a perguntas diferentes. O gráfico por **dia** revela se há variações de volume ao longo do período — quedas ou picos que podem sinalizar anomalias ou início de experimento. O gráfico por **hora do dia** expõe o ritmo de uso intradiário, útil para entender o comportamento do usuário e detectar se logs noturnos (horários atípicos) têm concentração anormal, o que indicaria lançamentos em lote ou artefatos técnicos. Usamos `px.bar` no primeiro (agrupamento manual por data via `groupby`) e `px.histogram` no segundo (distribuição contínua por hora).

### Os dados são igualmente completos para todo o período?

In [110]:
# Reutilizando a contagem diária criada no bloco anterior
daily_median = events_per_day['n_eventos'].median()  # mediana: referência robusta, não afetada por dias atípicos

# Limiar: dias com menos de 50% da mediana são tratados como incompletos
# Esse corte é conservador — captura apenas dias visivelmente abaixo da curva
limiar = daily_median * 0.5

print(f'Mediana diária de eventos : {daily_median:.0f}')
print(f'Limiar de incompletude    : {limiar:.0f} eventos/dia (qualquer dia com menos eventos que isso é suspeito de estar incompleto)')
print()

incomplete_days = events_per_day[events_per_day['n_eventos'] < limiar].copy()  # filtrando apenas dias abaixo do limiar

print(f'Dias com dados possivelmente incompletos: {len(incomplete_days)}')
if not incomplete_days.empty:  # ou seja, se houver pelo menos um dia incompleto
    print(incomplete_days[['date', 'n_eventos']].to_string(index=False))  # exibindo sem índice numérico para leitura limpa

Mediana diária de eventos : 16563
Limiar de incompletude    : 8282 eventos/dia (qualquer dia com menos eventos que isso é suspeito de estar incompleto)

Dias com dados possivelmente incompletos: 7
      date  n_eventos
2019-07-25          9
2019-07-26         31
2019-07-27         55
2019-07-28        105
2019-07-29        184
2019-07-30        412
2019-07-31       2030


> **Raciocínio:** A mediana é preferida à média como referência porque é resistente a outliers — um dia extraordinariamente movimentado não eleva artificialmente o patamar. O limiar de 50% da mediana é deliberadamente conservador: queremos capturar apenas dias que claramente ficam fora do padrão estabelecido, não dias com pequenas variações. Dias com volume baixo no início do log tipicamente ocorrem porque o experimento ainda estava sendo rampado ou porque eventos de usuários foram registrados com atraso (retroativamente), criando uma falsa cauda inicial no dataset.

### Qual período os dados realmente representam?

In [111]:
# Definindo a data de corte: o último dia identificado como incompleto

if not incomplete_days.empty:
    data_corte = incomplete_days['date'].max()  # data do último dia incompleto — tudo antes é descartado
    food_app_filtered = food_app_clean[food_app_clean['datetime'] > data_corte].copy()  # mantém apenas registros após o corte
else:
    # Se não há dias incompletos, o dataset completo já é o período real
    data_corte = None
    food_app_filtered = food_app_clean.copy()

# Exibe o período real após o corte
real_start = food_app_filtered['date'].min()                                   # primeira data do período válido
real_end     = food_app_filtered['date'].max()                                  # última data do período válido
real_period = (pd.to_datetime(real_end) - pd.to_datetime(real_start)).days    # duração em dias do período limpo

print(f'Data de corte            : {data_corte}')
print(f'Período real — início    : {real_start}')
print(f'Período real — fim       : {real_end}')
print(f'Duração do período real  : {real_period} dias')
print(f'Eventos no período real  : {len(food_app_filtered):,}')

Data de corte            : 2019-07-31 00:00:00
Período real — início    : 2019-07-31
Período real — fim       : 2019-08-07
Duração do período real  : 7 dias
Eventos no período real  : 242,917


> **Raciocínio:** Descartamos tudo que está **igual ou antes** do último dia incompleto (`>` em vez de `>=`) porque eventos do próprio dia de corte podem estar parcialmente registrados — manter a borda seria arriscado. Ao salvar o resultado em `food_app_filtered` (sem sobrescrever `food_app_clean`) preservamos o dataset limpo original para eventual reverificação. Esse filtro garante que as análises de funil e de teste A/B serão feitas sobre janelas de tempo uniformes, condição necessária para comparações justas entre os grupos de controle e experimento.

### Quantos eventos e usuários foram perdidos ao excluir os dados mais antigos?

In [112]:
# Eventos antes e depois do corte
events_before  = len(food_app_clean)                    # total de eventos no dataset limpo (antes do corte temporal)
events_after = len(food_app_filtered)                 # total de eventos após a remoção dos dias incompletos
missed_events = events_before - events_after       # diferença absoluta
pct_missed_events = missed_events / events_before * 100  # impacto percentual sobre o total

# Usuários antes e depois do corte
users_before  = food_app_clean['device_id'].nunique()     # usuários únicos antes do corte
users_after = food_app_filtered['device_id'].nunique()  # usuários únicos após o corte
missed_users = users_before - users_after        # diferença absoluta
pct_missed_users = missed_users / users_before * 100  # impacto percentual sobre o total

print('--- Impacto do corte temporal ---')
print(f'Eventos  antes : {events_before:>8,}')
print(f'Eventos  depois: {events_after:>8,}   | perdidos: {missed_events:,} ({pct_missed_events:.2f}%)')
print()
print(f'Usuários antes : {users_before:>8,}')
print(f'Usuários depois: {users_after:>8,}   | perdidos: {missed_users:,} ({pct_missed_users:.2f}%)')

--- Impacto do corte temporal ---
Eventos  antes :  243,713
Eventos  depois:  242,917   | perdidos: 796 (0.33%)

Usuários antes :    7,551
Usuários depois:    7,542   | perdidos: 9 (0.12%)


> **Raciocínio:** Comparamos as mesmas métricas (eventos e usuários únicos) antes e depois do corte para quantificar o custo da limpeza. O percentual de perda é o indicador mais importante: uma perda pequena valida a decisão de descartar os dias incompletos sem comprometer a representatividade da amostra. Uma perda grande exigiria reconsiderar a estratégia de corte.

### Os três grupos experimentais ainda estão presentes após o corte?

In [113]:
# Quantificando usuários únicos por grupo experimental no dataset filtrado
users_per_group = (
    food_app_filtered
    .groupby('exp_id')['device_id']   # agrupa por grupo e seleciona a coluna de usuário
    .nunique()                         # conta apenas IDs distintos dentro de cada grupo
    .rename('usuarios_unicos')
    .reset_index()
)

# Adiciona o percentual de cada grupo em relação ao total de usuários filtrados
users_per_group['% do total'] = (
    users_per_group['usuarios_unicos'] / users_per_group['usuarios_unicos'].sum() * 100
).round(2).astype(str) + '%'

print('Usuários únicos por grupo experimental (após corte):')
print(users_per_group.to_string(index=False))
print()

# Confirmação de que os três grupos estão presentes
expected_groups = {246, 247, 248}                             # grupos definidos no experimento
present_groups = set(users_per_group['exp_id'].values)       # grupos encontrados no dataset filtrado
missing_groups  = expected_groups - present_groups         # diferença: grupos que deveriam estar mas não estão

if not missing_groups:
    print('✓ Todos os três grupos experimentais (246, 247, 248) estão presentes.')
else:
    print(f'✗ Grupos ausentes após o corte: {missing_groups}')

Usuários únicos por grupo experimental (após corte):
 exp_id  usuarios_unicos % do total
    246             2485     32.95%
    247             2517     33.37%
    248             2540     33.68%

✓ Todos os três grupos experimentais (246, 247, 248) estão presentes.


> **Raciocínio:** O corte temporal descarta eventos com base em data — não em grupo. Ainda assim, é necessário verificar se os três grupos (246 e 247 de controle, 248 de teste) sobreviveram ao filtro, pois um grupo sub-representado nos dias iniciais poderia desaparecer completamente. Além da presença, a coluna `% do total` permite detectar desequilíbrios severos entre grupos — condição que comprometeria qualquer comparação estatística posterior. A comparação com `set` é a forma mais direta de checar se todos os grupos esperados estão no resultado.

## Funil de eventos

### Quais eventos estão nos registros e com que frequência ocorrem?

In [114]:
# Contando o total de ocorrências de cada tipo de evento no período filtrado
freq_events = (
    food_app_filtered
    .groupby('event_name')
    .size()                           # número de linhas (= ocorrências) por evento
    .rename('incidents_number')
    .sort_values(ascending=False)     # ordena do mais frequente para o menos frequente
    .reset_index()
)

# Calculando o percentual de cada evento em relação ao total de ocorrências
freq_events['% of total'] = (
    freq_events['incidents_number'] / freq_events['incidents_number'].sum() * 100
).round(2).astype(str) + '%'

print('Frequência de ocorrências por evento (período filtrado):')
print('\n', freq_events.to_string(index=False))

Frequência de ocorrências por evento (período filtrado):

              event_name  incidents_number % of total
       MainScreenAppear            118578     48.81%
     OffersScreenAppear             46707     19.23%
       CartScreenAppear             42560     17.52%
PaymentScreenSuccessful             34058     14.02%
               Tutorial              1014      0.42%


> **Raciocínio:** Cada linha do dataset é uma ocorrência de evento, portanto `groupby + size()` é o caminho direto para contar quantas vezes cada tipo aparece. Ordenar por frequência decrescente permite identificar imediatamente quais eventos dominam o log e quais são raros — informação essencial antes de construir o funil, pois eventos com volume muito baixo podem ser etapas opcionais ou ruído de coleta. O percentual contextualiza cada evento dentro do volume total.

### Quantos usuários realizaram cada ação? Qual a proporção em relação ao total?

In [115]:
# Total de usuários únicos no período filtrado — denominador para calcular as proporções
total_users_filtered = food_app_filtered['device_id'].nunique()

# Para cada evento, conta quantos usuários distintos o realizaram pelo menos uma vez
# nunique() sobre device_id garante que um usuário que repetiu o evento conta apenas uma vez
users_per_event = (
    food_app_filtered
    .groupby('event_name')['device_id']
    .nunique()                            # usuários únicos por evento
    .rename('unique_users')
    .sort_values(ascending=False)         # ordena do evento com mais usuários para o com menos
    .reset_index()
)

# Proporção: usuários que fizeram o evento ÷ total de usuários no período
users_per_event['% of users'] = (
    users_per_event['unique_users'] / total_users_filtered * 100
).round(2).astype(str) + '%'

print(f'Total de usuários no período filtrado: {total_users_filtered:,}')
print()
print('\nUsuários únicos por evento (ordenado por número de usuários):')
print('\n', users_per_event.to_string(index=False))

Total de usuários no período filtrado: 7,542


Usuários únicos por evento (ordenado por número de usuários):

              event_name  unique_users % of users
       MainScreenAppear          7429      98.5%
     OffersScreenAppear          4606     61.07%
       CartScreenAppear          3742     49.62%
PaymentScreenSuccessful          3542     46.96%
               Tutorial           845      11.2%


> **Raciocínio:** A diferença em relação à contagem anterior é sutil mas crítica: aqui interessa saber **quantos usuários distintos** realizaram cada ação, não quantas vezes ela ocorreu. Um mesmo usuário pode passar pela tela de pagamento três vezes — mas para o funil ele conta como um único usuário naquela etapa. Por isso usamos `nunique()` sobre `device_id` agrupado por evento, e não `size()`. O denominador da proporção é o total de usuários únicos no período filtrado — garantindo que a porcentagem reflita o alcance real de cada etapa dentro da base de usuários ativa.

### Em que ordem os eventos ocorrem? Todos fazem parte de uma única sequência?

In [116]:
# Reutilizando users_per_event (já ordenado por número de usuários decrescente)
# A lógica do funil é: quanto mais alto na jornada, mais usuários passaram por ali
# Portanto, eventos com MAIS usuários são os primeiros da sequência

funnel_order = users_per_event.copy()  # cópia para não alterar o DataFrame original

# Calculando a taxa de retenção de uma etapa para a próxima (step-over-step)
# Dividindo o número de usuários de cada linha pelo da linha anterior
funnel_order['retention_next_step'] = (
    funnel_order['unique_users'].shift(-1) / funnel_order['unique_users'] * 100
).round(2)

print('Eventos ordenados por número de usuários (candidatos à ordem do funil):')
print()
print(funnel_order.to_string(index=False))

Eventos ordenados por número de usuários (candidatos à ordem do funil):

             event_name  unique_users % of users  retention_next_step
       MainScreenAppear          7429      98.5%                62.00
     OffersScreenAppear          4606     61.07%                81.24
       CartScreenAppear          3742     49.62%                94.66
PaymentScreenSuccessful          3542     46.96%                23.86
               Tutorial           845      11.2%                  NaN


> **Raciocínio:** A premissa central do funil de conversão é que o volume de usuários diminui a cada etapa — nenhum passo posterior pode ter mais usuários do que o anterior. Por isso, ordenar os eventos pelo número de usuários únicos de forma decrescente é a maneira mais objetiva de inferir a sequência provável sem depender de conhecimento prévio sobre o produto.
>
> O campo `retention_next_step` mostra qual percentual dos usuários de cada etapa chegou à etapa seguinte. Quedas abruptas nessa taxa indicam gargalos na jornada. 
>
> **Sobre a sequência:** Os eventos do app formam uma jornada linear de compra — tela principal → oferta → carrinho → pagamento. O evento `Tutorial` destoa desse padrão: ele é exibido apenas para novos usuários em seu primeiro acesso e pode ocorrer em qualquer ponto da sessão, não sendo uma etapa da conversão. Por não fazer parte da sequência de compra, ele será **excluído do funil** nas análises seguintes, conforme orienta o enunciado.

### Funil de conversão: proporção de usuários entre etapas

In [117]:
# Sequência do funil — apenas os eventos da jornada de compra, sem o Tutorial
FUNNEL_STEPS = [
    'MainScreenAppear',   # etapa 1: tela principal — ponto de entrada de todos os usuários
    'OffersScreenAppear', # etapa 2: tela de ofertas — usuário demonstrou interesse
    'CartScreenAppear',   # etapa 3: tela do carrinho — usuário adicionou itens
    'PaymentScreenSuccessful',  # etapa 4: pagamento concluído — conversão final
]

# Para cada etapa, conta os usuários únicos que a realizaram ao menos uma vez
funnel_data = []
for step in FUNNEL_STEPS:
    n_users = (
        food_app_filtered[food_app_filtered['event_name'] == step]['device_id'].nunique()
    )  # filtra o evento e conta IDs distintos
    funnel_data.append({'step': step, 'users': n_users})

funnel_df = pd.DataFrame(funnel_data)

# Taxa de conversão step-over-step: usuários da etapa N ÷ usuários da etapa N-1
funnel_df['conv_prev_step'] = (
    funnel_df['users'] / funnel_df['users'].shift(1) * 100  # shift(1) desloca os valores uma linha para cima, de modo que a divisão N ÷ N-1 fica natural
).round(2)

# Taxa de conversão acumulada: usuários de cada etapa ÷ usuários da etapa 1 (topo do funil)
# Isso mostra a porcentagem de usuários que chegaram até aquela etapa em relação ao total que iniciou a jornada
funnel_df['conv_from_top'] = (
    funnel_df['users'] / funnel_df['users'].iloc[0] * 100
).round(2)

print('Funil de conversão (todos os usuários — período filtrado):')
print()
print(funnel_df.to_string(index=False))

# Visualização do funil em gráfico de barras horizontal
fig_funnel = px.funnel(
    funnel_df,
    x='users',   # eixo x: volume de usuários
    y='step',      # eixo y: nome da etapa
    title='Funil de eventos — todos os usuários'
)
fig_funnel.show()

Funil de conversão (todos os usuários — período filtrado):

                   step  users  conv_prev_step  conv_from_top
       MainScreenAppear   7429             NaN         100.00
     OffersScreenAppear   4606           62.00          62.00
       CartScreenAppear   3742           81.24          50.37
PaymentScreenSuccessful   3542           94.66          47.68


> **Raciocínio:** O funil é construído iterando sobre a lista `FUNNEL_STEPS`, que define explicitamente a ordem das etapas e exclui o `Tutorial`. Duas taxas são calculadas:
>
> - **`conv_prev_step`** (step-over-step): responde à pergunta do enunciado — "qual proporção dos usuários da etapa A chegou à etapa B?". É o indicador de atrito entre passos consecutivos — onde as pessoas estão desistindo.
> - **`conv_from_top`** (acumulada): mostra o alcance de cada etapa em relação ao topo do funil. Permite responder "de todos que abriram o app, quantos chegaram ao pagamento?".
>
> O `px.funnel` é o tipo de gráfico nativo do Plotly para esse padrão — as barras são automaticamente ordenadas e proporcionais, facilitando a leitura visual do estreitamento da jornada.

### Em qual etapa perde-se mais usuários?

In [118]:
# Calculando o número absoluto de usuários perdidos entre cada par de etapas consecutivas
funnel_df['missing_users'] = funnel_df['users'].shift(1) - funnel_df['users']  # etapa anterior - etapa atual = abandono

# Identificando a etapa com maior perda absoluta (ignorando a primeira linha, pois não tem anterior)
biggest_drop_idx  = funnel_df['missing_users'].idxmax()              # índice da linha com maior perda
biggest_drop_step = funnel_df.loc[biggest_drop_idx, 'step']       # nome da etapa que recebe os que abandonaram
prev_step         = funnel_df.loc[biggest_drop_idx - 1, 'step']   # etapa imediatamente anterior (onde o abandono ocorre)
biggest_drop_n    = int(funnel_df.loc[biggest_drop_idx, 'missing_users'])   # número absoluto de usuários perdidos
biggest_drop_pct  = round(100 - funnel_df.loc[biggest_drop_idx, 'conv_prev_step'], 2)  # % que não avançou

print('Perda de usuários por transição:')
print('\n', funnel_df[['step', 'users', 'missing_users', 'conv_prev_step']].to_string(index=False))
print(f'\n► Maior gargalo: {prev_step} → {biggest_drop_step}')
print(f'  Usuários perdidos: {biggest_drop_n:,} ({biggest_drop_pct}% não avançou para a próxima etapa)')

Perda de usuários por transição:

                    step  users  missing_users  conv_prev_step
       MainScreenAppear   7429            NaN             NaN
     OffersScreenAppear   4606         2823.0           62.00
       CartScreenAppear   3742          864.0           81.24
PaymentScreenSuccessful   3542          200.0           94.66

► Maior gargalo: MainScreenAppear → OffersScreenAppear
  Usuários perdidos: 2,823 (38.0% não avançou para a próxima etapa)


> **Raciocínio:** `conv_prev_step` já nos diz o que avançou; `missing_users` inverte a leitura para focar no que foi perdido — subtração simples entre a etapa anterior e a atual. `idxmax()` encontra o índice da maior perda absoluta automaticamente, sem depender de valores fixos hardcoded — se o funil mudar, o código continua funcionando. A perda percentual é o complemento de `conv_prev_step` (100% − taxa de avanço), tornando o gargalo imediatamente interpretável.

### Qual é a parcela de usuários que faz o caminho inteiro, desde o primeiro evento até o pagamento?

In [119]:
# Recuperando o número de usuários na primeira e na última etapa do funil
users_top    = funnel_df.loc[funnel_df['step'] == FUNNEL_STEPS[0],  'users'].values[0]
users_bottom = funnel_df.loc[funnel_df['step'] == FUNNEL_STEPS[-1], 'users'].values[0]
# .loc[máscara_booleana, 'coluna'] → filtra a linha do passo e extrai o valor de 'users' em um único acesso

# Calculando a taxa de conclusão ponta a ponta (primeira → última etapa)
full_journey_pct = round(users_bottom / users_top * 100, 2)

print(f'Usuários que iniciaram o funil ({FUNNEL_STEPS[0]}): {users_top:,}')
print(f'Usuários que concluíram o pagamento ({FUNNEL_STEPS[-1]}): {users_bottom:,}')
print(f'\n► {full_journey_pct}% dos usuários completou o caminho inteiro (do início ao pagamento)')

Usuários que iniciaram o funil (MainScreenAppear): 7,429
Usuários que concluíram o pagamento (PaymentScreenSuccessful): 3,542

► 47.68% dos usuários completou o caminho inteiro (do início ao pagamento)


> **Raciocínio:** A taxa de conclusão ponta a ponta é simplesmente `usuários no último passo / usuários no primeiro passo`. Usamos `FUNNEL_STEPS[0]` e `FUNNEL_STEPS[-1]` para tornar o cálculo agnóstico ao número de etapas — se o funil for redefinido, a lógica permanece correta. Essa métrica sintetiza o desempenho global do funil em um único número: a fração de usuários que, ao abrir o app, chegou a concluir um pagamento.

## Resultados do experimento

### Quantos usuários há em cada grupo?

In [120]:
# Contando usuários únicos por grupo experimental nos dados filtrados
group_sizes = (
    food_app_filtered
    .groupby('exp_id')['device_id']       # agrupa por grupo e seleciona coluna de usuários
    .nunique()                             # conta IDs únicos (elimina duplicatas dentro do grupo)
    .reset_index()
    .rename(columns={'device_id': 'users'})
)

# Calculando a proporção de cada grupo em relação ao total
group_sizes['pct_of_total'] = round(
    group_sizes['users'] / group_sizes['users'].sum() * 100, 2  # divide cada grupo pelo total e converte para %
)

# Convertendo exp_id para string: evitando que o eixo x seja tratado como numérico (o que geraria 245.5, 246.5 etc.)
group_sizes['exp_id'] = group_sizes['exp_id'].astype(str)

print(group_sizes.to_string(index=False))
print(f"\nTotal de usuários nos 3 grupos: {group_sizes['users'].sum():,}")

# Definindo o range do eixo y perto dos valores reais para ampliar a diferença visual entre grupos
y_min = group_sizes['users'].min()
y_max = group_sizes['users'].max()

# Exibindo barras para comparação visual entre grupos
fig_groups = px.bar(
    group_sizes,
    x='exp_id', y='users',
    text='users',                          # exibe o valor sobre cada barra
    title='Usuários únicos por grupo experimental',
    labels={'exp_id': 'Grupo', 'users': 'Usuários únicos'},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig_groups.update_traces(textposition='outside', width=0.35)   # width < 1 estreitando as barras
fig_groups.update_layout(
    showlegend=False,
    yaxis_range=[y_min * 0.97, y_max * 1.04],  # zoom no eixo y para evidenciar diferenças entre grupos
)
fig_groups.show()

exp_id  users  pct_of_total
   246   2485         32.95
   247   2517         33.37
   248   2540         33.68

Total de usuários nos 3 grupos: 7,542


> **Raciocínio:** Usamos `nunique()` sobre `device_id` dentro de cada grupo para obter usuários únicos — mesma lógica do funil. O gráfico de barras torna imediato verificar se os grupos têm tamanhos parecidos: grupos desequilibrados poderiam comprometer a validade estatística do teste.

### Teste A/A: há diferença estatisticamente significativa entre os grupos de controle 246 e 247?

In [122]:
from statsmodels.stats.proportion import proportions_ztest  # proportions_ztest está em statsmodels, não em scipy

alpha = 0.05  # nível de significância padrão

# Conjuntos de device_ids de cada grupo de controle
# Nota: exp_id é int em food_app_filtered (a conversão para str foi feita apenas em group_sizes, para o gráfico)
users_246 = set(food_app_filtered.loc[food_app_filtered['exp_id'] == 246, 'device_id'])
users_247 = set(food_app_filtered.loc[food_app_filtered['exp_id'] == 247, 'device_id'])

n_246 = len(users_246)  # total de usuários únicos no grupo 246
n_247 = len(users_247)  # total de usuários únicos no grupo 247

print(f'Grupo 246: {n_246:,} usuários  |  Grupo 247: {n_247:,} usuários')
print(f'\nα = {alpha}  (p-value < α → diferença estatisticamente significativa)\n')
print(f'{"Etapa":<30} {"Conv 246":>10} {"Conv 247":>10} {"p-value":>10} {"Sig?":>6}')
print('─' * 70)

for step in FUNNEL_STEPS:
    # Usuários de cada grupo que realizaram este evento
    step_users = set(food_app_filtered.loc[food_app_filtered['event_name'] == step, 'device_id'])
    x_246 = len(step_users & users_246)  # interseção: usuários do grupo 246 nesta etapa
    x_247 = len(step_users & users_247)  # interseção: usuários do grupo 247 nesta etapa

    # proportions_ztest([sucessos_A, sucessos_B], [total_A, total_B]) — retorna (z_stat, p_value)
    # sucesso = usuário do grupo que chegou a esta etapa; total = todos os usuários do grupo
    # o '_' descarta o z-statistic — só o p-value é necessário para a decisão
    _, p_value = proportions_ztest([x_246, x_247], [n_246, n_247])

    conv_246 = x_246 / n_246 * 100  # taxa de conversão do grupo 246 nesta etapa (em %)
    conv_247 = x_247 / n_247 * 100  # taxa de conversão do grupo 247 nesta etapa (em %)
    sig      = 'Sim ⚠' if p_value < alpha else 'Não'  # sinaliza se a diferença é estatisticamente significativa


    print(f'{step:<30} {conv_246:>9.2f}% {conv_247:>9.2f}% {p_value:>10.4f} {sig:>6}')    
    # imprime uma linha da tabela com alinhamento fixo para facilitar a leitura

Grupo 246: 2,485 usuários  |  Grupo 247: 2,517 usuários

α = 0.05  (p-value < α → diferença estatisticamente significativa)

Etapa                            Conv 246   Conv 247    p-value   Sig?
──────────────────────────────────────────────────────────────────────
MainScreenAppear                   98.67%     98.49%     0.5869    Não
OffersScreenAppear                 62.13%     60.63%     0.2744    Não
CartScreenAppear                   50.99%     49.23%     0.2131    Não
PaymentScreenSuccessful            48.29%     46.05%     0.1121    Não


> **Raciocínio:** O teste A/A compara dois grupos que **não deveriam** ser diferentes — ambos recebem a experiência padrão. Se encontrarmos diferença significativa (p-value < α), isso indica falha na atribuição aleatória, na coleta de dados ou no próprio mecanismo experimental, e o teste A/B não seria confiável.
>
> Usamos o **z-test para duas proporções** (`proportions_ztest`): para cada etapa do funil, comparamos a taxa de conversão do grupo 246 com a do 247. A hipótese nula (H₀) é que as proporções são iguais — se não rejeitarmos H₀ em nenhuma etapa, os grupos são estatisticamente equivalentes e o experimento está bem calibrado.
>
> Para verificar quantos usuários de cada grupo realizaram um determinado evento, precisamos cruzar duas listas: "quem está no grupo 246" e "quem fez esse evento". Converter cada lista em um `set` Python e usar o operador `&` (interseção) faz exatamente isso em uma única operação — retorna apenas os IDs que aparecem nas duas listas ao mesmo tempo. O resultado é o número de usuários do grupo que chegou àquela etapa, que é o numerador da taxa de conversão.